# ML Coding — Interview Prep

Self-contained implementations of the ML/LLM coding problems that tend to come up in frontier-lab interviews. Each section states the problem, implements it, then verifies with a minimal sanity test.

In [2]:
import numpy as np
import torch
import torch.nn.functional as F
import math

## Scaled dot-product & multi-head attention (NumPy)

Implement attention from scratch in NumPy.

- `softmax(x, axis)` — numerically stable softmax (subtract the max before exponentiating).
- `simple_attention(Q, K, V, mask)` — scaled dot-product attention. `Q, K, V` are `(B, H, T, d_k)`. Scale scores by `1/sqrt(d_k)`, apply the mask *before* the softmax, and return both the output `(B, H, T, d_k)` and the attention weights.
- `causal_mask(T)` — lower-triangular boolean mask so position `t` attends only to positions `<= t`.
- `multi_head_attention(x, W_q, W_k, W_v, W_o, H, causal)` — project `x` `(B, T, d)` into Q/K/V, split into `H` heads, run attention per head, concatenate the heads back, and apply the output projection `W_o`.

In [3]:
# Numpy version
import numpy as np

def softmax(x: np.ndarray, axis=-1):
    adj_x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(adj_x)
    sum_x = np.sum(exp_x, axis=axis, keepdims=True)
    return exp_x / sum_x

def simple_attention(Q, K, V, mask=None):
    # Q, K, V: (B, H, T, d_k)
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    attention = softmax(scores)
    output = attention @ V

    return output, attention

def causal_mask(T):
    mask = np.tril(np.ones((T, T), dtype=bool))
    mask = mask[None, None, :, :]
    return mask

def multi_head_attention(x, W_q, W_k, W_v, W_o, H, causal=False):
    B, T, d = x.shape
    d_k = d // H
    mask = None if not causal else causal_mask(T)
    
    Q, K, V = x @ W_q, x @ W_k, x @ W_v
    # rearrange
    Q = Q.reshape(B, T, H, d_k).transpose(0, 2, 1, 3)
    K = K.reshape(B, T, H, d_k).transpose(0, 2, 1, 3)
    V = V.reshape(B, T, H, d_k).transpose(0, 2, 1, 3)

    output, _ = simple_attention(Q, K, V, mask)

    # output: (B, H, T, d_k) -> (B, T, H, d_k) -> (B, T, d)
    output = output.transpose(0, 2, 1, 3).reshape(B, T, d)

    return output @ W_o

In [4]:
B, T, H, d = 2, 4, 4, 16

x = np.random.randn(B, T, d)
W_q = np.random.randn(d, d)
W_k = np.random.randn(d, d)
W_v = np.random.randn(d, d)
W_o = np.random.randn(d, d)

out = multi_head_attention(x, W_q, W_k, W_v, W_o, H, causal=True)
print(out.shape == (B, T, d), out.shape, "Yes out shape is as expected")          # shape contract

# attention rows must be a probability distribution
_, attn = simple_attention(
    *[np.random.randn(B, H, T, d // H) for _ in range(3)],
    mask=causal_mask(T))
print(np.allclose(attn.sum(-1), 1.0), "Yes, attention scores sum to one")             # softmax sanity
# causal check: position 0 attends only to itself
print(np.allclose(attn[..., 0, 1:], 0.0), "Yes, attention scores are 0 for masked")         # upper triangle is zero

True (2, 4, 16) Yes out shape is as expected
True Yes, attention scores sum to one
True Yes, attention scores are 0 for masked


## Attention & multi-head attention (PyTorch)

The same problem re-implemented in PyTorch. Watch the idiom differences from the NumPy version: `masked_fill(mask == 0, -inf)` for masking, `math.sqrt` for the scale, `.view` / `.transpose(1, 2)` / `.reshape` for the head reshaping, and `F.softmax(dim=-1)`.

In [5]:
def sdpa_pytorch(Q, K, V, mask=None):
    # Q, K, V: (B, H, T, d_k)
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attention = F.softmax(scores, dim=-1)
    z = attention @ V
    return z, attention

def causal_mask_pytorch(T):
    mask = torch.tril(torch.ones(T, T)).unsqueeze(0).unsqueeze(0)
    return mask

def mha_pytorch(x, W_q, W_k, W_v, W_o, H, causal=False):
    B, T, d = x.shape
    d_k = d // H
    mask = causal_mask_pytorch(T) if causal else None

    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v

    #rearrange
    Q = Q.view(B, T, H, d_k).transpose(1, 2)
    K = K.view(B, T, H, d_k).transpose(1, 2)
    V = V.view(B, T, H, d_k).transpose(1, 2)

    z, _ = sdpa_pytorch(Q, K, V, mask) # (B, H, T, d_k)
    # rearrange
    z = z.transpose(1, 2).reshape(B, T, d)
    o = z @ W_o

    return o

In [6]:
B, T, H, d = 2, 4, 2, 8
x = torch.randn(B, T, d)
Wq, Wk, Wv, Wo = [torch.randn(d, d) for _ in range(4)]

o = mha_pytorch(x, Wq, Wk, Wv, Wo, H, causal=True)
assert o.shape == (B, T, d)

# the test that actually matters:
_, attn = sdpa_pytorch(*[torch.randn(B,H,T,d//H) for _ in range(3)],
                       mask=causal_mask_pytorch(T))
assert torch.allclose(attn[..., 0, 1:], torch.zeros(T-1)), "future leaked"

## Token & sequence log-probabilities

Core building block for RLHF losses (DPO/GRPO). Given `logits (B, T, V)`, `labels (B, T)`, and a `loss_mask (B, T)`:

- `get_token_logprobs` — shift logits and labels by one for next-token prediction, take `log_softmax`, gather the log-prob of each true next token, and zero out masked positions. Returns `(B, T-1)`.
- `get_sequence_logprobs` — reduce token log-probs to one score per sequence, either `"sum"` or length-normalized `"mean"`. Clamp the mask count with `.clamp(min=1)` so fully-masked rows don't divide by zero.

In [7]:
def get_token_logprobs(logits, labels, loss_mask):
    # logits: (B, T, V)
    # labels: (B, T)
    # mask: (B, T)
    logits = logits[:, :-1, :]
    labels = labels[:, 1:]
    mask = loss_mask[:, 1:]

    logprobs = F.log_softmax(logits, dim=-1)

    # B, T-1
    token_logprobs = torch.gather(logprobs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)

    token_logprobs = token_logprobs * mask

    return token_logprobs

def get_sequence_logprobs(logits, labels, loss_mask, reduction="sum"):
    token_logprobs = get_token_logprobs(logits, labels, loss_mask)
    mask = loss_mask[:, 1:]
    if reduction == "sum":
        return token_logprobs.sum(dim=-1)
    elif reduction == "mean":
        return token_logprobs.sum(dim=-1) / mask.sum(dim=-1).clamp(min=1.0)
    else:
        raise NotImplementedError(f"{reduction} reduction not implemented")

## DPO loss

Direct Preference Optimization. Given the policy and reference log-probs for a chosen (positive) and rejected (negative) response, compute

$$\mathcal{L} = -\log \sigma\Big(\beta \big[(\log \pi_\theta^{+} - \log \pi_\text{ref}^{+}) - (\log \pi_\theta^{-} - \log \pi_\text{ref}^{-})\big]\Big)$$

averaged over the batch. The loss drops when the policy raises the chosen response's likelihood — relative to the reference — more than it raises the rejected one's.

In [8]:
def dpo_loss(positive_logprobs,
             positive_reference_logprobs,
             negative_logprobs,
             negative_reference_logprobs,
             beta):
    # (B,)
    positive_logratio = positive_logprobs - positive_reference_logprobs
    negative_logratio = negative_logprobs - negative_reference_logprobs

    margin = positive_logratio - negative_logratio
    logits = beta * margin

    loss = -F.logsigmoid(logits).mean()

    return loss

# make the policy prefer chosen MORE than reference does → loss should drop
good = dict(positive_logprobs=torch.tensor([-1.0]),
            positive_reference_logprobs=torch.tensor([-2.0]),   # policy ↑ on chosen
            negative_logprobs=torch.tensor([-3.0]),
            negative_reference_logprobs=torch.tensor([-2.0]),   # policy ↓ on rejected
            beta=0.1)
bad = dict(good, positive_logprobs=torch.tensor([-3.0]),
                 negative_logprobs=torch.tensor([-1.0]))        # flipped
print(dpo_loss(**good) < dpo_loss(**bad), "Works as expected")

tensor(True) Works as expected


In [9]:
B, T, V = 2, 5, 10
logits = torch.randn(B, T, V)
labels = torch.randint(0, V, (B, T))
loss_mask = torch.ones(B, T)
loss_mask[1] = 0                          # row 1 entirely masked

seq = get_sequence_logprobs(logits, labels, loss_mask, "sum")
assert seq.shape == (B,)
assert seq[1] == 0                         # masked row sums to 0

seq_mean = get_sequence_logprobs(logits, labels, loss_mask, "mean")
assert not torch.isnan(seq_mean).any()     # THIS fails without clamp

## GRPO loss

Group Relative Policy Optimization (as used in DeepSeek-R1). For each prompt, a group of `G` responses is sampled and scored. Given `logprobs`, `old_logprobs`, `ref_logprobs`, and `rewards (B, G)`:

- `compute_advantages(rewards)` — normalize rewards *within* each group: `(r - mean) / (std + eps)`. No learned value function.
- `grpo_loss` — PPO-style clipped surrogate: `min(ratio · A, clip(ratio, 1±eps) · A)` with `ratio = exp(logprobs - old_logprobs)`, plus a `β`-weighted KL penalty to the reference policy using the low-variance `k3` estimator `exp(Δ) - Δ - 1` where `Δ = ref_logprobs - logprobs`.

In [10]:
def compute_advantages(rewards, eps=1e-8):
    # rewards: (B, G)
    mean = rewards.mean(dim=-1, keepdim=True)
    std = rewards.std(dim=-1, keepdim=True)

    advantages = (rewards - mean) / (std + eps)

    return advantages

def grpo_loss(
    logprobs,
    old_logprobs,
    ref_logprobs,
    rewards,
    beta,
    clip_eps
):
    # logprobs: (B,)
    # rewards: (B, G)
    advantages = compute_advantages(rewards)

    logratio = logprobs - old_logprobs
    ratio = torch.exp(logratio)

    unclipped = advantages * ratio
    clipped = advantages * torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)
    policy_loss = -torch.min(unclipped, clipped)

    kl = torch.exp(ref_logprobs - logprobs) - (ref_logprobs - logprobs) - 1

    loss = (policy_loss + beta * kl).mean()

    return loss

## Cosine similarity + Top-k retrieval

Given `queries (Nq, d)` and `docs (Nd, d)`, implement `topk_cosine(queries, docs, k)` returning the indices `(Nq, k)` and scores `(Nq, k)` of the `k` most similar docs per query, sorted descending. L2-normalize both sides (with an eps for numerical safety), take the dot product to get the `(Nq, Nd)` cosine-similarity matrix, then `topk` along the doc axis.

In [13]:
def topk_cosine(queries, docs, k):
    # queries: (Nq, d)
    # docs: (Nd, d)
    # returns: indices (Nq, k), scores (Nq, k), sorted descending
    queries = queries / (torch.linalg.norm(queries, dim=-1, keepdim=True) + 1e-8)
    docs = docs / (torch.linalg.norm(docs, dim=-1, keepdim=True) + 1e-8)

    # (Nq, Nd)
    similarity_scores = queries @ docs.T
    topk_scores, topk_indices = torch.topk(similarity_scores, k=k, dim=-1)

    return topk_indices, topk_scores

In [14]:
Nq, Nd, d, k = 2, 5, 4, 3
q, docs = torch.randn(Nq, d), torch.randn(Nd, d)
idx, sc = topk_cosine(q, docs, k)
assert idx.shape == (Nq, k) and sc.shape == (Nq, k)
assert (idx >= 0).all() and (idx < Nd).all()      # indices are valid docs
assert (sc[:, 0] >= sc[:, -1]).all()               # descending

## Sampling

Given a 1-D `logits` vector of shape `(V,)` over the vocabulary, implement `sample_next(logits, temperature, top_k, top_p)` that returns a single sampled token id. Apply temperature, then top-k filtering, then top-p (nucleus) filtering, then sample from what remains.

In [31]:
def sample_next(logits, temperature, top_k, top_p):
    # logits: (B, V)
    if temperature == 0:
        return logits.argmax(dim=-1, keepdim=True)

    logits = logits / temperature

    if top_k is not None:
        v, _ = torch.topk(logits, k=top_k)
        logits[logits < v[:, [-1]]] = float('-inf')

    if top_p is not None:
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumprobs = torch.cumsum(sorted_probs, dim=-1)

        nucleus = cumprobs <= top_p
        nucleus[..., 0] = True

        k = nucleus.sum(dim=-1, keepdim=True)          # (B, 1)
        threshold = torch.gather(sorted_probs, dim=-1, index=k - 1)
        logits = torch.where(probs < threshold, float('-inf'), logits)

    probs = F.softmax(logits, dim=-1)
    
    pred = torch.multinomial(probs, num_samples=1)

    return pred

In [32]:
B, V = 3, 10
logits = torch.randn(B, V)
# greedy
assert torch.equal(sample_next(logits, 0, None, None), logits.argmax(-1, keepdim=True))
# top_k=1 → argmax
assert torch.equal(sample_next(logits, 1.0, 1, None), logits.argmax(-1, keepdim=True))
# top_p valid ids
out = sample_next(logits, 1.0, None, 0.9)
assert out.shape == (B, 1) and (out >= 0).all() and (out < V).all()

## Perplexity

Given logits of shape (T, V) and targets of shape (T,) (the true token id at each position), return the perplexity as a scalar.

In [34]:
def perplexity(logits, targets):
    logprobs = F.log_softmax(logits, dim=-1)
    selected_scores = torch.gather(logprobs, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
    mean_score = -selected_scores.mean()
    perplexity = torch.exp(mean_score)
    return perplexity

In [36]:
V = 10
logits = torch.zeros(5, V)                        # uniform over V
targets = torch.randint(0, V, (5,))
print(abs(perplexity(logits, targets).item() - V) < 1e-4, "perplexity works")   # should be 10.0

True perplexity works
